## Figure 4a

In [1]:
#!/usr/bin/env python3
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from matplotlib.lines import Line2D
import matplotlib.font_manager as fm

# ============================================================
# Fonts / plotting defaults
# ============================================================
font_paths = [
    "/home/gzu5140/Font/Arial.ttf",
    "/home/gzu5140/Font/Arial Bold.ttf",
    "/home/gzu5140/Font/Arial Italic.ttf",
    "/home/gzu5140/Font/Arial Bold Italic.ttf",
]
for fp in font_paths:
    try:
        fm.fontManager.addfont(fp)
        print("✔ Loaded font:", fp)
    except Exception as e:
        print("⚠️  Could not load:", fp, "|", e)

plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['font.family'] = "sans-serif"
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = "none"
plt.rcParams['mathtext.fontset'] = "cm"
plt.rcParams['axes.labelsize'] = 25
plt.rcParams['axes.titlesize'] = 26
plt.rcParams['xtick.labelsize'] = 24
plt.rcParams['ytick.labelsize'] = 24
plt.rcParams['legend.fontsize'] = 24
plt.rcParams['figure.dpi'] = 400
plt.rcParams['axes.grid'] = False

# ============================================================
# Config
# ============================================================
t1_fixed = 1
t2_values = list(range(1, 49))   # 1..48
time_col = 'time_step'

# Gene columns (Z, X, Y)
y_col  = 'gene_1_mRNA'  # Z
x1_col = 'gene_2_mRNA'  # X
x2_col = 'gene_3_mRNA'  # Y

# Metrics to compute / plot
METRICS_TO_PLOT = ["Spearman(x_t1,y_t2)", "Spearman(y_t1,x_t2)"]

# Label the genes as Z, X, Y (for legend text)
GENE_LABELS = {y_col: "Z", x1_col: "X", x2_col: "Y"}

# Pair order (undirected pairs) and labels:
#   X-Y, Z-Y, Z-X
PAIR_ORDER  = [
    (x1_col, x2_col),   # X-Y
    (y_col,  x2_col),   # Z-Y
    (y_col,  x1_col),   # Z-X
]
PAIR_LABELS = {(a, b): f"{GENE_LABELS[a]}-{GENE_LABELS[b]}" for (a, b) in PAIR_ORDER}

# Colors
COL_ZX = "#3E8ED0"
COL_ZY = "#F5B700"
COL_XY = "#D73027"
COL_YX = "#00A676"

# ============================================================
# AUTO-DISCOVERY: folders + filename keywords only
# include_all = AND logic (must contain ALL keywords)
# ============================================================
SEARCH_ROOTS = [
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/three_gene_sim_all_variants",
]

# IMPORTANT:
# Put tokens separately so it's truly AND:
# e.g. ["fan_out_additive_new", "n_2"] means filename must contain BOTH.
MOTIF_RULES = {
    "Fan-out": {
        "include_all": ["Fan_out_additive", "df_rows"],
        "exclude_any": ["k_on0"],
        "max_files": 20,   # set int (e.g., 20) to cap newest files
    },
    "Feed-forward Loop": {
        "include_all": ["Feed_forward_additive", "df_rows"],
        "exclude_any": ["k_on0"],
        "max_files": 20,
    },
    "Regulated Mutual": {
        "include_all": ["Mutual_regulation_additive", "df_rows"],
        "exclude_any": ["k_on0"],
        "max_files": 20,
    },
}

def discover_motif_files(roots, rules, pattern="df*.csv", recursive=False, sort_by="mtime"):
    """
    Discover df*.csv files in roots, then assign them to motifs using ONLY filename checks.

    rules[motif]:
      - include_all: list[str] (filename must contain ALL substrings)  [AND]
      - exclude_any: list[str] (filename must contain NONE of these)
      - max_files: int|None    (keep newest N if sort_by='mtime')
    """
    candidates = []
    for root in roots:
        if recursive:
            candidates += glob.glob(os.path.join(root, "**", pattern), recursive=True)
        else:
            candidates += glob.glob(os.path.join(root, pattern))

    # de-dup and keep real files
    candidates = [p for p in dict.fromkeys(candidates) if p and os.path.isfile(p)]

    def _key(p):
        if sort_by == "name":
            return os.path.basename(p).lower()
        return os.path.getmtime(p)  # default: mtime

    out = {}
    for motif, r in rules.items():
        inc_all = [s.lower() for s in r.get("include_all", [])]
        exc_any = [s.lower() for s in r.get("exclude_any", [])]
        max_files = r.get("max_files", None)

        hits = []
        for p in candidates:
            name = os.path.basename(p).lower()

            # AND filter: must contain ALL required substrings
            if inc_all and not all(k in name for k in inc_all):
                continue

            # exclusions: must contain NONE of these
            if exc_any and any(k in name for k in exc_any):
                continue

            hits.append(p)

        if sort_by == "name":
            hits = sorted(hits, key=_key)
        else:
            hits = sorted(hits, key=_key, reverse=True)  # newest first

        if isinstance(max_files, int) and max_files > 0:
            hits = hits[:max_files]

        out[motif] = hits
        print(f"[discover] {motif}: {len(hits)} files")
        if hits:
            print("  example:", os.path.basename(hits[0]))

    return out

MOTIF_FILES = discover_motif_files(
    SEARCH_ROOTS,
    MOTIF_RULES,
    pattern="df*.csv",
    recursive=False,   # set True if your files are in subfolders
    sort_by="mtime",
)

# ============================================================
# Helpers
# ============================================================
def _spearman(a, b):
    """Spearman rank correlation with pairwise finite masking."""
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3:
        return np.nan
    r, _ = spearmanr(a[mask], b[mask])
    return float(r)

def detect_twins_scheme(df: pd.DataFrame):
    """
    Decide which replicate labels are twin A and twin B for this file.

    - If replicates include 0 and 1  -> A = 0, B = 1
    - Else if replicates include 1 and 2 (and no 0) -> A = 1, B = 2
    - Otherwise, raise an error.
    """
    reps = set(df["replicate"].dropna().unique())
    if {0, 1}.issubset(reps):
        return 0, 1
    elif {1, 2}.issubset(reps) and 0 not in reps:
        return 1, 2
    else:
        raise ValueError(f"Cannot infer twin scheme from replicates: {sorted(reps)}")

# ============================================================
# Twins: cross-time Spearman (twin A @ t1, twin B @ t2)
# with HALF the clones used for cross-twin pairs
# ============================================================
def spearman_cross_twins_per_file_like_matrix(
    filepath,
    t1,
    t2,
    time_col=time_col,
    y_col=y_col,
    x1_col=x1_col,
    x2_col=x2_col,
    rep_t1=None,
    rep_t2=None,
    type_comparison="twin",  # "twin" or "random"
) -> pd.DataFrame:
    """
    Compute cross-time Spearman correlations using *half* the clones as real twins.

    For each undirected gene pair (g1, g2) in PAIR_ORDER, return:
      - Spearman(x_t1,y_t2): g1(t1, A) vs g2(t2, B)
      - Spearman(y_t1,x_t2): g2(t1, A) vs g1(t2, B)

    If type_comparison == "random", break twin pairing by permuting the t2 rows
    within the half-sampled set.
    """
    df = pd.read_csv(filepath)

    # decide repA/repB for THIS file
    if rep_t1 is None or rep_t2 is None:
        repA, repB = detect_twins_scheme(df)
    else:
        repA, repB = rep_t1, rep_t2

    base_cols = ["clone_id", "replicate", y_col, x1_col, x2_col]

    df_t1 = df.loc[(df[time_col] == t1) & (df["replicate"] == repA), base_cols].copy()
    df_t2 = df.loc[(df[time_col] == t2) & (df["replicate"] == repB), base_cols].copy()

    # average duplicates (if any)
    if df_t1.duplicated(subset=["clone_id"]).any():
        df_t1 = df_t1.groupby("clone_id", as_index=False).mean(numeric_only=True)
    if df_t2.duplicated(subset=["clone_id"]).any():
        df_t2 = df_t2.groupby("clone_id", as_index=False).mean(numeric_only=True)

    # strict twin safety check
    clones_t1 = set(df_t1["clone_id"])
    clones_t2 = set(df_t2["clone_id"])
    if type_comparison == "twin":
        if clones_t1 != clones_t2:
            only_t1 = sorted(clones_t1 - clones_t2)
            only_t2 = sorted(clones_t2 - clones_t1)
            print("[error] Twin mismatch between t1 and t2 for file:", filepath)
            print(f"  t1 (rep={repA}) clones: {len(clones_t1)}")
            print(f"  t2 (rep={repB}) clones: {len(clones_t2)}")
            if only_t1:
                print("  present only at t1 (first few):", only_t1[:10])
            if only_t2:
                print("  present only at t2 (first few):", only_t2[:10])
            raise ValueError("Mismatch in clone_id sets between t1 and t2 for twin comparison")

    rename_t1 = {col: f"{col}_t1" for col in [y_col, x1_col, x2_col]}
    rename_t2 = {col: f"{col}_t2" for col in [y_col, x1_col, x2_col]}
    df_t1 = df_t1.rename(columns=rename_t1)
    df_t2 = df_t2.rename(columns=rename_t2)

    merged = df_t1.merge(df_t2, on="clone_id", how="inner")
    if merged.empty:
        cols = ["gene_pair", "Spearman(x_t1,y_t2)", "Spearman(y_t1,x_t2)"]
        return pd.DataFrame(columns=cols).set_index("gene_pair")

    # HALF-sampling
    n_total = len(merged)
    if n_total >= 2:
        n_keep = n_total // 2
        idx = np.random.permutation(n_total)[:n_keep]
        merged = merged.iloc[idx].reset_index(drop=True)

    # random: break pairing
    if type_comparison == "random":
        n = len(merged)
        perm = np.random.permutation(n)
        merged_rand = merged.copy()
        for col in [f"{y_col}_t2", f"{x1_col}_t2", f"{x2_col}_t2"]:
            merged_rand[col] = merged[col].values[perm]
        used_df = merged_rand
    else:
        used_df = merged

    rows = []
    for (g1, g2) in PAIR_ORDER:
        x_forward = used_df[f"{g1}_t1"].values
        y_forward = used_df[f"{g2}_t2"].values
        r_forward = _spearman(x_forward, y_forward)

        x_reverse = used_df[f"{g2}_t1"].values
        y_reverse = used_df[f"{g1}_t2"].values
        r_reverse = _spearman(x_reverse, y_reverse)

        rows.append({
            "gene_pair": PAIR_LABELS[(g1, g2)],
            "Spearman(x_t1,y_t2)": r_forward,
            "Spearman(y_t1,x_t2)": r_reverse,
        })

    return pd.DataFrame(rows).set_index("gene_pair")

# ============================================================
# Twins: build tidy table over t2 for all files in a motif
# ============================================================
def build_tidy_for_files(files, motif_name, t1, t2_values):
    files = [p for p in dict.fromkeys(files) if p and os.path.isfile(p)]
    if not files:
        print(f"[warn] No valid files for motif: {motif_name}")
        return pd.DataFrame(columns=["motif","file","t1","t2","gene_pair","metric","value"])

    print(f"[info] {motif_name}: using {len(files)} twin files")
    records = []
    for fp in files:
        base = os.path.basename(fp)
        for t2 in t2_values:
            try:
                tbl = spearman_cross_twins_per_file_like_matrix(
                    fp, t1, t2,
                    time_col=time_col,
                    y_col=y_col,
                    x1_col=x1_col,
                    x2_col=x2_col,
                    type_comparison="twin",
                )
            except Exception as e:
                print(f"[warn] {motif_name} | {base} | t2={t2}: {e}")
                continue

            for gp, row in tbl.iterrows():
                for metric in METRICS_TO_PLOT:
                    val = row.get(metric, np.nan)
                    records.append({
                        "motif":     motif_name,
                        "file":      base,
                        "t1":        int(t1),
                        "t2":        int(t2),
                        "gene_pair": gp,
                        "metric":    metric,
                        "value":     None if pd.isna(val) else float(val),
                    })

    return pd.DataFrame.from_records(records)

# ============================================================
# Regular: population gene-gene Spearman (full population, per time)
# ============================================================
def build_tidy_regular_for_files(files, motif_name, t_values=None, t1=None):
    files = [p for p in dict.fromkeys(files) if p and os.path.isfile(p)]
    if not files:
        print(f"[warn] No valid files for motif (regular): {motif_name}")
        return pd.DataFrame(columns=["motif","file","t1","t2","gene_pair","metric","value"])

    print(f"[info-pop] {motif_name}: computing population corr at each time")
    records = []

    for fp in files:
        base = os.path.basename(fp)
        try:
            df = pd.read_csv(fp)
        except Exception as e:
            print(f"[warn-pop] {motif_name} | {base}: {e}")
            continue

        try:
            repA, repB = detect_twins_scheme(df)
        except ValueError as e:
            print(f"[warn-pop] {motif_name} | {base}: {e}")
            continue

        if t_values is None:
            times = np.sort(df[time_col].dropna().unique())
        else:
            times = np.array(sorted(set(int(t) for t in t_values)))

        for t in times:
            sub = df[(df["replicate"] == repB) & (df[time_col] == t)].dropna(
                subset=[x1_col, x2_col, y_col]
            )
            if sub.empty:
                continue

            rho_xy = _spearman(sub[x1_col], sub[x2_col])
            rho_zy = _spearman(sub[y_col],  sub[x2_col])
            rho_zx = _spearman(sub[y_col],  sub[x1_col])

            for (g1, g2) in PAIR_ORDER:
                gp = PAIR_LABELS[(g1, g2)]
                if gp == "X-Y":
                    rho = rho_xy
                elif gp == "Z-Y":
                    rho = rho_zy
                elif gp == "Z-X":
                    rho = rho_zx
                else:
                    continue

                records.append({
                    "motif":     motif_name,
                    "file":      base,
                    "t1":        int(t),
                    "t2":        int(t),
                    "gene_pair": gp,
                    "metric":    "Spearman_same_time",
                    "value":     float(rho),
                })

    return pd.DataFrame.from_records(records)

# ============================================================
# Aggregate
# ============================================================
def aggregate_mean_std(tidy: pd.DataFrame) -> pd.DataFrame:
    if tidy.empty:
        return pd.DataFrame(columns=["motif","metric","gene_pair","t2","mean","std","n"])
    return (
        tidy
        .groupby(["motif", "metric", "gene_pair", "t2"], as_index=False)
        .agg(mean=('value', 'mean'),
             std =('value', 'std'),
             n   =('value', 'count'))
    )

# ============================================================
# Plotter: ONE panel per motif, overlay rho_hat twins + rho_pop
# ============================================================
def plot_three_motifs_panels(agg_twin: pd.DataFrame,
                             agg_reg: pd.DataFrame,
                             motifs_order=None,
                             outpath=None):
    if motifs_order is None:
        motifs_order = list(MOTIF_FILES.keys())

    fig, axes = plt.subplots(1, 3, sharey=True, figsize=(22, 6.5))

    for ax, motif in zip(axes, motifs_order):
        sub_twin = agg_twin[agg_twin["motif"] == motif]
        sub_reg  = agg_reg[agg_reg["motif"] == motif]

        if sub_twin.empty:
            print(f"[plot] No twin rows for motif={motif}")
            continue

        zx = sub_twin[(sub_twin["gene_pair"] == "Z-X") &
                      (sub_twin["metric"] == "Spearman(x_t1,y_t2)")].sort_values("t2")
        if not zx.empty:
            t = zx["t2"].values
            m = zx["mean"].values
            sd = zx["std"].values
            ax.plot(t, m, lw=2.5, color=COL_ZX, ls='-')
            if np.all(np.isfinite(sd)):
                ax.fill_between(t, m - sd, m + sd, color=COL_ZX, alpha=0.15, linewidth=0)

        zy = sub_twin[(sub_twin["gene_pair"] == "Z-Y") &
                      (sub_twin["metric"] == "Spearman(x_t1,y_t2)")].sort_values("t2")
        if not zy.empty:
            t = zy["t2"].values
            m = zy["mean"].values
            sd = zy["std"].values
            ax.plot(t, m, lw=2.5, color=COL_ZY, ls='-')
            if np.all(np.isfinite(sd)):
                ax.fill_between(t, m - sd, m + sd, color=COL_ZY, alpha=0.15, linewidth=0)

        xy = sub_twin[(sub_twin["gene_pair"] == "X-Y") &
                      (sub_twin["metric"] == "Spearman(x_t1,y_t2)")].sort_values("t2")
        if not xy.empty:
            t = xy["t2"].values
            m = xy["mean"].values
            sd = xy["std"].values
            ax.plot(t, m, lw=2.5, color=COL_XY, ls='-')
            if np.all(np.isfinite(sd)):
                ax.fill_between(t, m - sd, m + sd, color=COL_XY, alpha=0.15, linewidth=0)

        yx = sub_twin[(sub_twin["gene_pair"] == "X-Y") &
                      (sub_twin["metric"] == "Spearman(y_t1,x_t2)")].sort_values("t2")
        if not yx.empty:
            t = yx["t2"].values
            m = yx["mean"].values
            sd = yx["std"].values
            ax.plot(t, m, lw=2.5, color=COL_YX, ls='-')
            if np.all(np.isfinite(sd)):
                ax.fill_between(t, m - sd, m + sd, color=COL_YX, alpha=0.15, linewidth=0)

        # if not sub_reg.empty:
        #     reg_xy = sub_reg[(sub_reg["gene_pair"] == "X-Y") &
        #                      (sub_reg["metric"] == "Spearman_same_time")].sort_values("t2")
        #     if not reg_xy.empty:
        #         t = reg_xy["t2"].values
        #         m = reg_xy["mean"].values
        #         sd = reg_xy["std"].values
        #         ax.plot(t, m, lw=2.0, color="black", ls="--")
        #         if np.all(np.isfinite(sd)):
        #             ax.fill_between(t, m - sd, m + sd, color="0.7", alpha=0.3, linewidth=0)

        ax.set_title(motif)
        ax.set_xlabel(r"$t_2\ \mathrm{[hours]}$")
        ax.set_ylim(0.0, 0.2)
        ax.tick_params(labelsize=16, width=1.4)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.4)
            spine.set_edgecolor("black")

    axes[0].set_ylabel(r"twin cross-correlation")

    legend_elements = [
        Line2D([], [], color=COL_ZX, lw=2.5, ls='-',
               label=r"Z→X $\hat{\rho}^{\dagger}_{\,z(t_{1}=1) \;\rightarrow\; x(t_{2})}$"),
        Line2D([], [], color=COL_ZY, lw=2.5, ls='-',
               label=r"Z→Y $\hat{\rho}^{\dagger}_{\,z(t_{1}=1) \;\rightarrow\; y(t_{2})}$"),
        Line2D([], [], color=COL_XY, lw=2.5, ls='-',
               label=r"X→Y $\hat{\rho}^{\dagger}_{\,x(t_{1}=1) \;\rightarrow\; y(t_{2})}$"),
        Line2D([], [], color=COL_YX, lw=2.5, ls='-',
               label=r"Y→X $\hat{\rho}^{\dagger}_{\,y(t_{1}=1) \;\rightarrow\; x(t_{2})}$"),
        #Line2D([], [], color="black", lw=2.0, ls='--',
               #label=r"X→Y $\rho_{xy}$"),
    ]
    axes[2].legend(handles=legend_elements, loc="upper right", frameon=False, fontsize=16)

    fig.tight_layout()

    if outpath is not None:
        fig.savefig(outpath, format="pdf", bbox_inches="tight",
                    facecolor="none", edgecolor="none", transparent=True)
        print(f"[saved] {outpath}")

    return fig

# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    all_tidy_twin = []
    for motif, files in MOTIF_FILES.items():
        tidy_motif = build_tidy_for_files(files, motif, t1_fixed, t2_values)
        if not tidy_motif.empty:
            all_tidy_twin.append(tidy_motif)

    if not all_tidy_twin:
        raise SystemExit("[abort] No twin data collected. Check SEARCH_ROOTS / MOTIF_RULES.")

    tidy_twin = pd.concat(all_tidy_twin, ignore_index=True)
    agg_twin  = aggregate_mean_std(tidy_twin)

    all_tidy_reg = []
    for motif, files in MOTIF_FILES.items():
        tidy_reg_motif = build_tidy_regular_for_files(files, motif, t_values=t2_values, t1=t1_fixed)
        if not tidy_reg_motif.empty:
            all_tidy_reg.append(tidy_reg_motif)

    if not all_tidy_reg:
        raise SystemExit("[abort] No regular-corr data collected. Check SEARCH_ROOTS / MOTIF_RULES.")

    tidy_reg = pd.concat(all_tidy_reg, ignore_index=True)
    agg_reg  = aggregate_mean_std(tidy_reg)

    fig_out = "/home/gzu5140/Cross_correlation_triplet_fig4_new_additive.pdf"
    plot_three_motifs_panels(agg_twin, agg_reg, outpath=fig_out)


✔ Loaded font: /home/gzu5140/Font/Arial.ttf
✔ Loaded font: /home/gzu5140/Font/Arial Bold.ttf
✔ Loaded font: /home/gzu5140/Font/Arial Italic.ttf
✔ Loaded font: /home/gzu5140/Font/Arial Bold Italic.ttf
[discover] Fan-out: 20 files
  example: df_rows_0_0_0_13122025_152146_ncells_6000_Fan_out_additive_2_11_e17abef9.csv
[discover] Feed-forward Loop: 20 files
  example: df_rows_0_0_0_13122025_152324_ncells_6000_Feed_forward_additive_1_9_a7b44308.csv
[discover] Regulated Mutual: 20 files
  example: df_rows_0_0_0_15122025_004231_ncells_6000_Mutual_regulation_additive_0_7_eea19e70.csv
[info] Fan-out: using 20 twin files
[info] Feed-forward Loop: using 20 twin files


## Figure 4b

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import wilcoxon, ttest_rel
import os

In [ ]:
import matplotlib.font_manager as fm
# ============================================================
# Fonts / style
# ============================================================
font_paths = [
    "/home/gzu5140/Font/Arial.ttf",
    "/home/gzu5140/Font/Arial Bold.ttf",
    "/home/gzu5140/Font/Arial Italic.ttf",
    "/home/gzu5140/Font/Arial Bold Italic.ttf",
]

for fp in font_paths:
    try:
        fm.fontManager.addfont(fp)
        print("✔ Loaded font:", fp)
    except Exception as e:
        print("⚠️  Could not load:", fp, "|", e)

# ==== LaTeX + SVG text mode (Illustrator-safe) ====
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42  # For PDF export
plt.rcParams['ps.fonttype'] = 42   # For PostScript (EPS) export
plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['font.family'] = "sans-serif"
plt.rcParams['svg.fonttype'] = "none"
plt.rcParams['mathtext.fontset'] = "cm"
plt.rcParams['axes.labelsize'] = 18     # x/y labels
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['xtick.labelsize'] = 12     # x-axis tick labels
plt.rcParams['ytick.labelsize'] = 12    # y-axis tick labels
plt.rcParams['legend.fontsize'] = 12    # legend text

In [ ]:
#Path to the plot data
path_to_plot_data = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_4_data/New_additive/"
#/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_4_data/New_additive//twins_random_zscore_summary.csv
path_to_plots = "/home/gzu5140/Keerthana_b1042/grnInference/plots/figure_4/"
os.makedirs(path_to_plots, exist_ok = True)

In [ ]:
# Load the data
random_correlation_path = f"{path_to_plot_data}/random_matrix_results.csv"
twin_correlation_t1_path= f"{path_to_plot_data}/twin_t1_matrix_results.csv"
random_correlation_data = pd.read_csv(random_correlation_path)
twin_correlation_t1_data = pd.read_csv(twin_correlation_t1_path)

In [ ]:
import pandas as pd

# Load data about twin correlation, z-scores and median of random pair correlations
loaded_df = pd.read_csv(f"{path_to_plot_data}/twins_random_zscore_summary.csv")

# Helper function: extract list for a specific network + metric
def extract_list(df, net, metric, gene_pair_set):
    return df[(df.network_type == net) & (df.metric == metric) & (df.gene_pair == gene_pair_set)]["values"].tolist()

# === Reconstruct all 9 lists ===

Fan_out_medians  = extract_list(loaded_df, "Fan_out", "medians", "('gene_2', 'gene_3')")
Fan_out_z_scores = extract_list(loaded_df, "Fan_out", "z_scores", "('gene_2', 'gene_3')")
Fan_out_threshold = extract_list(loaded_df, "Fan_out", "z_threshold_list_neg", "('gene_2', 'gene_3')")

Feed_forward_medians = extract_list(loaded_df, "Feed_forward", "medians", "('gene_2', 'gene_3')")
Feed_forward_z_scores = extract_list(loaded_df, "Feed_forward", "z_scores","('gene_2', 'gene_3')")
Feed_forward_threshold = extract_list(loaded_df, "Feed_forward", "z_threshold_list_neg", "('gene_2', 'gene_3')")

Mutual_regulation_medians = extract_list(loaded_df, "Mutual_regulation", "medians", "('gene_2', 'gene_3')")
Mutual_regulation_z_scores = extract_list(loaded_df, "Mutual_regulation", "z_scores", "('gene_2', 'gene_3')")
Mutual_regulation_threshold = extract_list(loaded_df, "Mutual_regulation", "z_threshold_list_neg", "('gene_2', 'gene_3')")


In [ ]:
# random-pair correlation medians
random_correlation_medians = [
    Fan_out_medians,
    Feed_forward_medians,
    Mutual_regulation_medians,
]
threshold_lists = [
    Fan_out_threshold,
    Feed_forward_threshold,
    Mutual_regulation_threshold
]
colors = ['#194a9e', '#006937', '#ee9127']
light_colors = ['#194a9e', '#006937', '#ee9127']
# Collect data for plotting
all_data = []
all_colors = []
positions = []
thresholds_1pct = []
z_scores = []
z_scores = [Fan_out_z_scores, Feed_forward_z_scores, Mutual_regulation_z_scores]
pos = 1
for i, sim_type in enumerate(sim_types_to_plot):
    
    # 1. Use random correlation medians
    random_vals = np.array(random_correlation_medians[i])
    random_vals = random_vals[~np.isnan(random_vals)]
    
    # 2. Get twin correlation data from CSV
    twin_sim = twin_correlation_t1_data[twin_correlation_t1_data['sim_type'] == sim_type]
    twin_vals = twin_sim[gene_pair].values  # Adjust column name if needed
    
    if len(random_vals) > 0 and len(twin_vals) > 0:
        # Calculate 1st percentile threshold from random medians        
        # Add to plotting data (random medians first, then twin)
        all_data.extend([random_vals, twin_vals])
        all_colors.extend([light_colors[i], colors[i]])
        positions.extend([pos, pos + 1])       
        pos += 3  # Space between groups
    else:
        print(f"Warning: {sim_type} missing data - Random: {len(random_vals)}, Twin: {len(twin_vals)}")


In [ ]:
fig_zscore = plt.figure(figsize=(8, 6))
ax_zscore = fig_zscore.add_subplot(111)

# Create z-score boxplot using the z_scores calculated above
z_box = ax_zscore.boxplot(z_scores, patch_artist=False, showfliers=True)

# Style z-score boxplot with matching colors
for i, patch in enumerate(z_box['boxes']):
    patch.set_color(colors[i])
    patch.set_clip_on(False)

for i, median in enumerate(z_box['medians']):
    median.set_color(colors[i])
    median.set_linewidth(2)
    median.set_clip_on(False)

for i, whisker in enumerate(z_box['whiskers']):
    whisker.set_color(colors[i//2])
    whisker.set_linewidth(1.5)
    whisker.set_clip_on(False)

for i, cap in enumerate(z_box['caps']):
    cap.set_color(colors[i//2])
    cap.set_linewidth(1.5)
    cap.set_clip_on(False)

# Style outliers
for i, flier in enumerate(z_box['fliers']):
    if len(flier.get_data()[0]) > 0:
        flier.set_markerfacecolor("none")
        flier.set_markeredgecolor(colors[i])
        flier.set_markersize(4)

# Add z-score thresholds
#ax_zscore.axhline(-1*z_score_threshold, linestyle="--", color="black", linewidth=1, alpha=0.7, label="z-score threshold")
ax_zscore.axhline(z_score_threshold, linestyle="--", color="black", linewidth=1, alpha=0.7)


# Set labels and styling
ax_zscore.set_xticks([1,2,3])
ax_zscore.set_xticklabels(["Fan out", "Feed forward", "Mutual regulation"])
ax_zscore.set_ylabel("Z-score", fontsize=12)
ax_zscore.tick_params(labelsize=10)
# ax_zscore.legend(fontsize=10)

# Save z-score plot
# plt.savefig(f"{path_to_plots}/zscore_boxplot_{gene_pair}.pdf", 
#            format="pdf", 
#            bbox_inches='tight',
#            facecolor='none',
#            edgecolor='none',
#            transparent=True)
# plt.savefig(f"{path_to_plots}/zscore_boxplot_{gene_pair}.svg", 
#            format="svg", 
#            bbox_inches='tight',
#            facecolor='none',
#            edgecolor='none',
#            transparent=True)
# plt.savefig(f"{path_to_plots}/zscore_boxplot_{gene_pair}.png", 
#            format="png", 
#            bbox_inches='tight',
#            facecolor='none',
#            edgecolor='none',
#            transparent=True)

plt.show()

## Figure 4c